# Learning journeys with FastPath, as a predictor

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jose-alvarado-guzman/oulad/blob/main/notebooks/aga_fastpath_journeys.ipynb)

`aga_outcome_prediction.ipynb` embeds students with FastRP, which sees *who* they are connected
to. This one embeds them with **FastPath**, which sees the **shape of the journey**: what they
did, in what order, how long ago, and how hard. FastPath is built for sequences — clickstreams,
customer journeys, event logs — and OULAD's VLE data is a clickstream.

The catch is that the loaded graph has no sequence in it. `REVIEWED_MATERIAL` is a direct
student→material edge carrying a date, with nothing linking one interaction to the next. So this
notebook **builds an event chain first**:

```
(:Student)-[:FIRST_INTERACTION]->(:Interaction)-[:NEXT_INTERACTION]->(:Interaction)-> ...
                                       |
                                 [:OF_MATERIAL]-> (:EducationalMaterial)
```

One `:Interaction` per (student, material, day), chained in date order. FastPath turns each
chain into a vector, and that vector is then judged the only way that answers whether it is
useful: as a **feature for a classifier**, scored on held-out F1 and accuracy against the same
target and the same baseline as the FastRP notebook.

Three variants are trained — journey embedding alone, volume alone, and both — because a single
number in isolation says nothing about whether the embedding earned its keep.

## Before you start

The usual secrets: `NEO4J_URI`, `NEO4J_USERNAME`, `NEO4J_PASSWORD`, `AURA_CLIENT_ID`,
`AURA_CLIENT_SECRET`, `AURA_PROJECT_ID`, with *Notebook access* on. Run
[`oulad_data_load.ipynb`](oulad_data_load.ipynb) first if the graph is not loaded.

> **This notebook writes to your database.** Step 5 creates one `:Interaction` per interaction
> — about 281,000 for the default module, 3.2M for `FFF` — plus two properties on each
> `Student`. **Step 15 deletes all of it.** Nothing else is modified; the OULAD graph is only
> read.

> **A session is billed compute**, separate from AuraDB. Step 15 deletes it; the TTL in step 8
> is only a backstop.

## FastPath is in preview

No Aura-native docs yet. The closest public reference is the Snowflake Graph Analytics
documentation, which covers the same algorithm:
<https://neo4j.com/docs/snowflake-graph-analytics/current/algorithms/fastpath/>

Its Python surface is still moving. Every parameter below was read off
`graphdatascience==2.0a5` directly, and several were renamed from earlier alphas
(`dimension` → `embedding_dimension`, `max_elapsed_time` → `lookback_horizon`,
`num_elapsed_times` → `num_time_anchors`, `time_node_property` →
`event_node_time_property`, `output_time` → `observation_time`, `decay_factor` →
`decay_rate`). Examples written against 2.0a1 will not run as-is.

## 1. Setup

Re-running resets the checkout to `origin/main`, discarding local changes.

In [ ]:
import os
import subprocess
import sys

REPO_URL = 'https://github.com/jose-alvarado-guzman/oulad.git'
REPO_DIR = '/content/oulad'

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

def run(*command):
    result = subprocess.run(command, text=True, capture_output=True)
    print((result.stdout + result.stderr).strip())
    result.check_returncode()

if IN_COLAB:
    if os.path.isdir(os.path.join(REPO_DIR, '.git')):
        run('git', '-C', REPO_DIR, 'fetch', '--depth', '1', 'origin', 'main')
        run('git', '-C', REPO_DIR, 'reset', '--hard', 'origin/main')
        run('git', '-C', REPO_DIR, 'clean', '-fd')
    else:
        run('git', 'clone', '--depth', '1', REPO_URL, REPO_DIR)
    run('git', '-C', REPO_DIR, 'log', '-1', '--format=%h %ad %s', '--date=short')
    print()
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '-r',
         os.path.join(REPO_DIR, 'requirements-aga.txt')], check=True)
    print('Dependencies installed.')
else:
    REPO_DIR = os.getcwd()
    while REPO_DIR != '/' and not os.path.isdir(os.path.join(REPO_DIR, '.git')):
        REPO_DIR = os.path.dirname(REPO_DIR)
    print('Local kernel; assuming requirements-aga.txt is installed.')
    print('Repository root:', REPO_DIR)

## 2. Imports

In [ ]:
import os
import sys
from datetime import timedelta

REPO_DIR = '/content/oulad' if os.path.isdir('/content/oulad') else REPO_DIR
SRC_DIR = os.path.join(REPO_DIR, 'src')
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

for name in [m for m in sys.modules if m == 'oulad' or m.startswith('oulad.')]:
    del sys.modules[name]

import matplotlib.pyplot as plt
import pandas as pd
from neo4j import GraphDatabase
from graphdatascience.session import (
    AlgorithmCategory, AuraAPICredentials, DbmsConnectionInfo, GdsSessions,
    SessionMemory)

import graphdatascience
from oulad.credentials import (
    AGA_SECRETS, ETL_SECRETS, MissingCredentialsError, aura_instance_id, load_credentials)
from oulad.logger import get_logger

print('graphdatascience', graphdatascience.__version__)
print('repository      ', REPO_DIR)

## 3. Credentials and the database connection

In [ ]:
logger = get_logger(REPO_DIR)

try:
    print('resolved from:', load_credentials(logger, required=ETL_SECRETS + AGA_SECRETS))
except MissingCredentialsError as error:
    raise SystemExit(f'\n{error}\n\nAdd the missing secrets in the sidebar, switch on '
                     'Notebook access, then re-run this cell.')

NEO4J_URI = os.environ['NEO4J_URI']
NEO4J_USERNAME = os.environ['NEO4J_USERNAME']
NEO4J_PASSWORD = os.environ['NEO4J_PASSWORD']
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE') or None
AURA_INSTANCE_ID = aura_instance_id(logger)

# liveness_check_timeout is not optional for this notebook. FastPath and training
# can leave the driver idle for tens of minutes, and a pooled connection that has
# gone stale surfaces as "Unable to retrieve routing information" or "Failed to
# read from defunct connection" in whichever cell happens to run next -- which on
# one run was the cleanup step, leaving 597,336 orphan Interaction nodes behind.
# With it set, an idle connection is verified before reuse.
driver = GraphDatabase.driver(
    NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD),
    liveness_check_timeout=30, max_connection_lifetime=600)
driver.verify_connectivity()
print('connected to AuraDB, instance', AURA_INSTANCE_ID)

sessions = GdsSessions(api_credentials=AuraAPICredentials(
    os.environ['AURA_CLIENT_ID'], os.environ['AURA_CLIENT_SECRET'],
    os.environ['AURA_PROJECT_ID']))

## 4. Choose the scope and read the time span

One module at a time. The chain is one node per interaction, so the module size *is* the
write size — and every event has to fit inside FastPath's lookback window, which the dates
below determine.

| Module | Students | Events | Avg chain |
| --- | --- | --- | --- |
| `AAA` | 702 | 280,990 | 400 |
| `GGG` | 2,359 | 281,277 | 119 |
| `EEE` | 2,634 | 758,220 | 288 |
| `CCC` | 3,852 | 884,889 | 230 |
| `BBB` | 6,484 | 1,139,085 | 176 |
| `DDD` | 5,407 | 1,866,156 | 345 |
| `FFF` | 6,799 | 3,248,703 | 478 |

`GGG` is the default: enough students for the grouping to mean something, the smallest write,
and short chains. `MAX_STUDENTS` caps the build while you are trying things out — set it to
`None` for the whole module.

OULAD dates are days relative to the module start and can be negative (material viewed before
the course opened), so they are shifted to begin at 0.

In [ ]:
MODULE = os.environ.get('OULAD_MODULE', 'GGG')
MAX_STUDENTS = None        # e.g. 200 for a quick trial, None for the whole module
CUTOFF_DAY = 90            # the recommended cutoff; step 15 requires it.
                           # Set to None for the whole journey, which step 14's
                           # sweep needs -- but the untruncated model is hindsight,
                           # not a predictor, so it is not what gets persisted.

SPAN_QUERY = '''
MATCH (s:Student)-[r:REVIEWED_MATERIAL]->(m:EducationalMaterial)<-[:HAS_MATERIAL]-(c:Course)
WHERE c.codeModule = $module
RETURN count(DISTINCT s) AS students, count(r) AS events,
       min(r.date) AS minDate, max(r.date) AS maxDate
'''
# Deliberately NOT scoped to $module. Activity-type vocabularies differ by
# module -- GGG has 7 types, BBB 12, EEE 11, out of 20 in the dataset -- and a
# per-module enumeration makes activityTypeId mean different things in different
# modules: 0 is 'forumng' in GGG and 'dualpane' in EEE. FastPath treats it as a
# categorical, so embeddings built that way do not share a space and a model
# trained on one module cannot validly score another. A fixed global vocabulary
# is what makes the persisted model in step 15 transferable.
TYPES_QUERY = '''
MATCH (m:EducationalMaterial)
RETURN DISTINCT m.activityType AS activityType ORDER BY activityType
'''

with driver.session(database=NEO4J_DATABASE) as session:
    span = session.run(SPAN_QUERY, module=MODULE).single()
    activity_types = [r['activityType'] for r in session.run(TYPES_QUERY)]

# GDS node properties must be numeric, so the activity type is encoded as an
# integer. FastPath's event_node_categorical_properties expects that too --
# hence event_node_ignored_category being an int in its signature.
TYPE_IDS = {name: index for index, name in enumerate(activity_types)}

SHIFT = -min(0, span['minDate'])          # move day 0 to the earliest event
last_day = span['maxDate'] if CUTOFF_DAY is None else CUTOFF_DAY
OBSERVATION_TIME = float(last_day + SHIFT + 1)
LOOKBACK_HORIZON = int(OBSERVATION_TIME) + 10   # must exceed the oldest elapsed time
NUM_TIME_ANCHORS = 20

print(f"module {MODULE}: {span['students']:,} students, {span['events']:,} events")
print(f"dates {span['minDate']} to {span['maxDate']}, shifted by +{SHIFT} "
      f'-> 0 to {span["maxDate"] + SHIFT}')
if CUTOFF_DAY is not None:
    print(f'TRUNCATED: only events on or before day {CUTOFF_DAY} '
          f'(shifted {CUTOFF_DAY + SHIFT}) will be chained')
print(f'observation_time {OBSERVATION_TIME:.0f}, lookback_horizon {LOOKBACK_HORIZON}, '
      f'{NUM_TIME_ANCHORS} time anchors '
      f'({LOOKBACK_HORIZON / NUM_TIME_ANCHORS:.1f} days per anchor)')
print(f'\nglobal activity-type vocabulary ({len(TYPE_IDS)} types, shared by every '
      f'module so embeddings are comparable):')
print(f'  {TYPE_IDS}')

## 5. Build the event chain

**This writes to your database.** One `:Interaction` per (student, material, day) — verified
unique in the source, so no aggregation is needed — chained in date order with ties broken by
material id so the sequence is deterministic.

Batched by student, because one transaction for a whole module is a bad idea. Re-running is
safe: students that already have a chain are skipped, so an interrupted build resumes.

Each `:Interaction` carries `day` (shifted), `clicks`, `activityTypeId`, `seq`, and `module`.
The `module` tag is what makes the scope and the cleanup in step 14 precise.

In [ ]:
BATCH = 100

CANDIDATES_QUERY = '''
MATCH (s:Student)-[r:REVIEWED_MATERIAL]->(:EducationalMaterial)<-[:HAS_MATERIAL]-(c:Course)
WHERE c.codeModule = $module AND NOT (s)-[:FIRST_INTERACTION]->(:Interaction)
  AND ($cutoff IS NULL OR r.date <= $cutoff)
RETURN DISTINCT s.id AS studentId ORDER BY studentId
'''

BUILD_QUERY = '''
UNWIND $studentIds AS studentId
MATCH (s:Student {id: studentId})-[r:REVIEWED_MATERIAL]->(m:EducationalMaterial)
      <-[:HAS_MATERIAL]-(c:Course)
WHERE c.codeModule = $module
  AND ($cutoff IS NULL OR r.date <= $cutoff)
WITH s, m, r ORDER BY r.date, m.id
WITH s, collect({material: m, day: r.date + $shift, clicks: r.sumClick,
                 typeId: $typeIds[m.activityType]}) AS events
UNWIND range(0, size(events) - 1) AS i
WITH s, i, events[i] AS event
// the material has to be bound to its own variable: a node pulled out of a map
// cannot be used directly inside a CREATE pattern
WITH s, i, event.material AS material, event.day AS day,
     event.clicks AS clicks, event.typeId AS typeId
CREATE (ev:Interaction {module: $module, studentId: s.id, seq: i, day: day,
                        clicks: clicks, activityTypeId: typeId,
                        // FastPath reads numeric event features as a vector, so the
                        // click count has to be a list even though it is one number.
                        // Logged: raw totals are heavily skewed.
                        features: [log(toFloat(clicks) + 1.0)]})
CREATE (ev)-[:OF_MATERIAL]->(material)
WITH s, ev ORDER BY ev.seq
WITH s, collect(ev) AS chain
// list elements need binding too, for the same reason as the material above
WITH s, chain, chain[0] AS firstEvent
CREATE (s)-[:FIRST_INTERACTION]->(firstEvent)
WITH chain
UNWIND range(0, size(chain) - 2) AS j
WITH chain[j] AS previous, chain[j + 1] AS following
CREATE (previous)-[:NEXT_INTERACTION]->(following)
RETURN count(*) AS links
'''

with driver.session(database=NEO4J_DATABASE) as session:
    pending = [r['studentId'] for r in session.run(
        CANDIDATES_QUERY, module=MODULE, cutoff=CUTOFF_DAY)]

if MAX_STUDENTS is not None:
    pending = pending[:MAX_STUDENTS]

print(f'{len(pending):,} students still need a chain')
built = 0
for start in range(0, len(pending), BATCH):
    chunk = pending[start:start + BATCH]
    with driver.session(database=NEO4J_DATABASE) as session:
        session.run(BUILD_QUERY, studentIds=chunk, module=MODULE,
                    shift=SHIFT, typeIds=TYPE_IDS, cutoff=CUTOFF_DAY).consume()
    built += len(chunk)
    if built % (BATCH * 5) == 0 or built == len(pending):
        print(f'  {built:,}/{len(pending):,} students', flush=True)
print('chain built' if pending else 'nothing to do, chain already present')

# The classifier needs its target and its baseline feature as numeric node
# properties. They are written onto Student here and removed again in step 16;
# the alternative is computing them in the projection query, which is awkward
# because that query starts from the chain rather than from the student.
LABEL_QUERY = '''
MATCH (s:Student)-[:WAS_REGISTERED]->(:StudentRegistration)-[cc:CONTAINS_COURSE]->(c:Course)
WHERE c.codeModule = $module
WITH DISTINCT s, cc.finalResult AS finalResult
OPTIONAL MATCH (s)-[r:REVIEWED_MATERIAL]->(:EducationalMaterial)<-[:HAS_MATERIAL]-(c2:Course)
WHERE c2.codeModule = $module
  // the same window as the chain: a truncated sequence paired with whole-module
  // click totals would be a truncated feature next to a hindsight one
  AND ($cutoff IS NULL OR r.date <= $cutoff)
WITH s, finalResult, coalesce(sum(r.sumClick), 0) AS clicks
SET s.passed = CASE WHEN finalResult IN $passResults THEN 1 ELSE 0 END,
    s.logClicks = log(toFloat(clicks) + 1.0)
RETURN count(*) AS labelled
'''
PASS_RESULTS = ['Pass', 'Distinction']
with driver.session(database=NEO4J_DATABASE) as session:
    labelled = session.run(LABEL_QUERY, module=MODULE, cutoff=CUTOFF_DAY,
                           passResults=PASS_RESULTS).single()['labelled']
print(f'labelled {labelled:,} students with passed and logClicks')

# Assessment SUBMISSION -- whether a student handed anything in -- is behaviour,
# not outcome, and is observable at the cutoff. Scores stay excluded: they
# determine finalResult and would be leakage. See docs/assessment-submission.md.
#
# Three exclusions, each of which would otherwise corrupt the feature:
#   isBanked = 1 is a score carried from a previous sitting, not engagement
#   exams have no due date (11 of 206), so they cannot inform a cutoff decision
#   isNaN on dateUnregistration is load-bearing -- 22,521 of 32,593 values are
#     NaN and none are null, so IS NULL alone drops 69% of the cohort
SUBMISSION_QUERY = '''
MATCH (c:Course {codeModule: $module})-[:HAS_ASSESSMENT]->(a:Assessment)
WHERE a.date IS NOT NULL AND NOT isNaN(a.date)
  AND a.assessmentType <> 'Exam'
  AND ($cutoff IS NULL OR a.date <= $cutoff)
WITH c, collect(a) AS due, min(a.date) AS firstDue
WITH c, due, size(due) AS nDue,
     [x IN due WHERE x.date = firstDue] AS firstAssessments
MATCH (s:Student)-[:WAS_REGISTERED]->(:StudentRegistration)-[:CONTAINS_COURSE]->(c)
OPTIONAL MATCH (s)-[w:WAS_ASSESSED_IN]->(a2:Assessment)
WHERE a2 IN due
  AND ($cutoff IS NULL OR w.dateSubmitted <= $cutoff)
  AND (w.isBanked IS NULL OR w.isBanked = 0)
WITH s, nDue, firstAssessments,
     count(DISTINCT a2) AS nSubmitted,
     avg(w.dateSubmitted - a2.date) AS lateness,
     count(DISTINCT CASE WHEN a2 IN firstAssessments THEN a2 END) AS gotFirst
SET s.submissionRate = CASE WHEN nDue = 0 THEN 1.0
                            ELSE toFloat(nSubmitted) / nDue END,
    s.missedAll   = CASE WHEN nDue > 0 AND nSubmitted = 0 THEN 1 ELSE 0 END,
    s.missedFirst = CASE WHEN gotFirst = 0 THEN 1 ELSE 0 END,
    s.meanLateness = coalesce(lateness, 0.0)
RETURN count(*) AS scored, max(nDue) AS assessmentsDue
'''
with driver.session(database=NEO4J_DATABASE) as session:
    got = session.run(SUBMISSION_QUERY, module=MODULE, cutoff=CUTOFF_DAY).single()
print(f"submission features on {got['scored']:,} students "
      f"({got['assessmentsDue']} assessments due by the cutoff)")
if not got['assessmentsDue']:
    print('  WARNING: no assessment is due by this cutoff, so every submission '
          'feature is constant and contributes nothing.')


## 6. Check the chain before trusting it

Three things worth confirming, because a broken chain would still embed — just wrongly:

- one `FIRST_INTERACTION` per student with a chain, and no student with two,
- as many `NEXT_INTERACTION` links as events minus students, since each chain of length *n*
  contributes *n-1* links,
- days along a chain never decrease.

In [ ]:
CHECK_QUERY = '''
MATCH (i:Interaction {module: $module})
WITH count(i) AS events
MATCH (:Student)-[f:FIRST_INTERACTION]->(:Interaction {module: $module})
WITH events, count(f) AS firsts
MATCH (:Interaction {module: $module})-[n:NEXT_INTERACTION]->()
RETURN events, firsts, count(n) AS nexts
'''
ORDER_QUERY = '''
MATCH (a:Interaction {module: $module})-[:NEXT_INTERACTION]->(b:Interaction)
WHERE b.day < a.day
RETURN count(*) AS outOfOrder
'''
with driver.session(database=NEO4J_DATABASE) as session:
    counts = session.run(CHECK_QUERY, module=MODULE).single()
    out_of_order = session.run(ORDER_QUERY, module=MODULE).single()['outOfOrder']

expected_nexts = counts['events'] - counts['firsts']
print(f"events {counts['events']:,}, chains {counts['firsts']:,}, "
      f"next links {counts['nexts']:,} (expected {expected_nexts:,})")
print('link count consistent:', counts['nexts'] == expected_nexts)
print('interactions out of date order:', out_of_order)
if counts['nexts'] != expected_nexts or out_of_order:
    raise SystemExit('The chain is not well formed; embedding it would be meaningless. '
                     'Delete it with step 16 and rebuild.')
print('\nchain looks sound')

## 7. A journey, in the raw

Worth seeing what FastPath is being handed before it turns into 128 numbers.

In [ ]:
SAMPLE_QUERY = '''
MATCH (s:Student)-[:FIRST_INTERACTION]->(first:Interaction {module: $module})
WITH s, first ORDER BY s.id LIMIT 1
MATCH path = (first)-[:NEXT_INTERACTION*0..14]->(ev:Interaction)
WITH s, ev ORDER BY ev.seq LIMIT 15
MATCH (ev)-[:OF_MATERIAL]->(m:EducationalMaterial)
RETURN s.id AS studentId, ev.seq AS seq, ev.day AS day,
       m.activityType AS activityType, ev.activityTypeId AS typeId, ev.clicks AS clicks
ORDER BY seq
'''
with driver.session(database=NEO4J_DATABASE) as session:
    sample = pd.DataFrame(session.run(SAMPLE_QUERY, module=MODULE).data())
print(f'first 15 events of one student\'s journey')
print(sample.to_string(index=False))

## 8. Size and open the session

In [ ]:
COUNT_QUERY = '''
MATCH (i:Interaction {module: $module})
WITH count(i) AS events
MATCH (s:Student)-[:FIRST_INTERACTION]->(:Interaction {module: $module})
WITH events, count(DISTINCT s) AS students
MATCH (:Interaction {module: $module})-[n:NEXT_INTERACTION]->()
RETURN events, students, events + count(n) AS relationships
'''
with driver.session(database=NEO4J_DATABASE) as session:
    sized = session.run(COUNT_QUERY, module=MODULE).single()

node_count = sized['events'] + sized['students']
relationship_count = sized['relationships']
print(f"{sized['students']:,} students + {sized['events']:,} interactions "
      f'= {node_count:,} nodes, {relationship_count:,} relationships')

memory = sessions.estimate(
    node_count=node_count,
    relationship_count=relationship_count,
    algorithm_categories=[AlgorithmCategory.NODE_EMBEDDING,
                          AlgorithmCategory.SIMILARITY,
                          AlgorithmCategory.COMMUNITY_DETECTION],
    node_label_count=2,          # Student, Interaction
    node_property_count=4,       # id, day, clicks, activityTypeId
)
print('estimated memory:', memory)

# sessions.estimate() under-sizes FastPath. For this chain -- 113k nodes, 220k
# relationships -- it returned m_2GB and FastPath aborted mid-run with "The job
# ran out of memory". The cascade is worse than the cause: with no embedding
# written, training then fails with "Node properties [journeyEmbedding] do not
# exist in the graph", which points at the pipeline rather than at memory.
# So the estimate is treated as a floor to raise, not a size to trust.
FLOOR = [(400_000, SessionMemory.m_8GB),
         (1_000_000, SessionMemory.m_16GB),
         (float('inf'), SessionMemory.m_32GB)]
required = next(m for limit, m in FLOOR if node_count < limit)
if memory != required:
    print(f'overriding {memory} -> {required} for {node_count:,} nodes '
          '(estimate() does not account for FastPath)')
    memory = required

SESSION_NAME = f"oulad-fastpath-{os.environ['AURA_CLIENT_ID'][:8]}"
gds = sessions.get_or_create(
    session_name=SESSION_NAME,
    memory=memory,
    db_connection=DbmsConnectionInfo(
        aura_instance_id=AURA_INSTANCE_ID,
        username=NEO4J_USERNAME, password=NEO4J_PASSWORD, database=NEO4J_DATABASE),
    ttl=timedelta(hours=2),
)
print('session ready:', SESSION_NAME)

## 9. Project the chain

Both chain relationship types in one query. The source of a row is a `Student` for
`FIRST_INTERACTION` and an `Interaction` for `NEXT_INTERACTION`, so the property map asks for
the union of both — a node simply has no value for the keys that do not apply to it.

Everything FastPath reads has to be numeric, which is why `activityTypeId` is projected rather
than `activityType`.

In [ ]:
GRAPH_NAME = 'oulad-journeys'

PROJECTION_QUERY = '''
MATCH (src)-[r:FIRST_INTERACTION|NEXT_INTERACTION]->(tgt:Interaction)
WHERE tgt.module = $module
RETURN gds.graph.project.remote(src, tgt, {
    sourceNodeLabels: labels(src),
    targetNodeLabels: labels(tgt),
    sourceNodeProperties: src { .id, .day, .clicks, .activityTypeId, .features,
                                .passed, .logClicks,
                                .submissionRate, .missedAll, .missedFirst, .meanLateness },
    targetNodeProperties: tgt { .day, .clicks, .activityTypeId, .features },
    relationshipType: type(r)
})
'''

gds.graph.project.cypher(
    graph_name=GRAPH_NAME, query=PROJECTION_QUERY,
    query_parameters={'module': MODULE}, overwrite=True)
G = gds.graph.get(GRAPH_NAME)
print(f'projected {G.node_count():,} nodes and {G.relationship_count():,} relationships')
print('node properties        :', G.node_properties())
print('relationship properties:', G.relationship_properties())

for label, needed in [('Interaction', {'day', 'activityTypeId', 'features'}),
                      ('Student', {'passed', 'logClicks', 'submissionRate',
                                   'missedAll', 'missedFirst', 'meanLateness'})]:
    missing = needed - set(G.node_properties().get(label, []))
    if missing:
        raise SystemExit(f'{label} is missing {missing} in the projection; FastPath would '
                         'either fail or read defaults. Check the projection query.')
print('\nthe properties FastPath needs are present')

## 10. FastPath embeddings

Each student's chain becomes one vector. The parameters worth understanding:

- **`observation_time`** is the vantage point. Elapsed time is measured back from here, and
  events at or after it are excluded — so it is set past the last event.
- **`lookback_horizon`** is how far back to look; it must exceed the oldest elapsed time or the
  earliest events fall outside the window and are dropped.
- **`num_time_anchors`** buckets that window. Twenty anchors over ~296 days is a fortnight
  each: enough to tell "worked steadily" from "crammed at the end".
- **`event_node_categorical_properties`** is what the events are *made of* — the kind of
  material touched.
- **`event_node_feature_vector_property`** is *how much* each event was worth. This carries
  the logged click count.
- **`random_seed`** is fixed so the embedding is reproducible.

> The feature vector was missing from the first version of this notebook, and its absence
> mattered: `sumClick` was copied onto the event nodes and projected into the session, then
> never passed to the algorithm. A page opened once and a page hammered fifty times were the
> same event. Any earlier result should be read as a different configuration, not as a verdict
> on FastPath.

In [ ]:
EMBEDDING_PROPERTY = 'journeyEmbedding'

embedding = gds.fast_path.mutate(
    G,
    base_node_label='Student',
    event_node_label='Interaction',
    mutate_property=EMBEDDING_PROPERTY,
    embedding_dimension=128,
    lookback_horizon=LOOKBACK_HORIZON,
    num_time_anchors=NUM_TIME_ANCHORS,
    event_node_categorical_properties=['activityTypeId'],
    event_node_feature_vector_property='features',   # click intensity
    event_node_time_property='day',
    first_relationship_type='FIRST_INTERACTION',
    next_relationship_type='NEXT_INTERACTION',
    observation_time=OBSERVATION_TIME,
    smoothing_window=2,
    smoothing_rate=10.0 / LOOKBACK_HORIZON,
    random_seed=42,
)
print(embedding)

## 11. Does the journey embedding predict the outcome?

The question a clustering score cannot answer. Louvain modularity says how cleanly the
similarity graph splits; it says nothing about whether the split is *useful*. So the embedding
goes into a **node classification pipeline** instead, and is judged on held-out F1 and accuracy
— the same measures, on the same module, as `aga_outcome_prediction.ipynb` uses for FastRP.

Three variants, because only the comparison is informative:

| variant | what it answers |
| --- | --- |
| journey embedding only | does the *sequence* carry predictive signal at all? |
| volume only | what does one logged aggregate already achieve? |
| journey + volume | does sequence add anything on top of volume? |

The embedding is already on the Student nodes from step 10, so no node-property step is needed
inside the pipeline — `select_features` names it directly.

In [ ]:
def train_variant(features, suffix):
    """Train the pipeline over `features` and return its held-out metrics."""
    pipeline_name = f'fastpath-pipeline-{suffix}'
    model_name = f'fastpath-model-{suffix}'
    for drop in (lambda: gds.model.get(model_name).drop(),
                 lambda: gds.pipeline.node_classification.get(pipeline_name).drop()):
        try:
            drop()
        except Exception:
            pass

    pipe, _ = gds.pipeline.node_classification.create(pipeline_name)
    pipe.select_features(features)
    pipe.configure_split(test_fraction=0.3, validation_folds=4)
    pipe.add_logistic_regression(penalty=(0.001, 1.0), max_epochs=300)
    pipe.add_random_forest(max_depth=(4, 16), number_of_decision_trees=200)

    model, _ = pipe.train(
        G, model_name=model_name, metrics=['F1_MACRO', 'ACCURACY'],
        target_property='passed', target_node_labels=['Student'], random_seed=42)
    scores = model.metrics() or {}
    method = (model.best_parameters() or {}).get('methodName', '?')
    return model, {m: v.get('test') for m, v in scores.items()
                   if isinstance(v, dict)}, method

VARIANTS = {
    'journey embedding only': ([EMBEDDING_PROPERTY], 'journey'),
    'volume only':            (['logClicks'], 'volume'),
    'journey + volume':       ([EMBEDDING_PROPERTY, 'logClicks'], 'both'),
}

models, results, winners = {}, {}, {}
for label, (features, suffix) in VARIANTS.items():
    print(f'\ntraining: {label}  features={features}', flush=True)
    model, scores, method = train_variant(features, suffix)
    models[label], results[label], winners[label] = model, scores, method
    print(f'  {scores}  (winner: {method})', flush=True)

## 12. The comparison

The row that matters is **journey + volume against volume only**. If the sequence embedding
carries information a single aggregate does not, that gap is where it shows up.

A threshold baseline is included too — predict pass above the median click count, no model at
all — scored over the same students the models see, since a student with no interactions has no
chain and so appears in neither.

> ### Read this before believing the number
>
> With `CUTOFF_DAY = None` the embedding sees each student's **whole** journey, right up to the
> end of the presentation. That includes *when the activity stopped* — and for a student who
> withdrew, when the activity stopped is very nearly the label itself. A strong score here is a
> real retrospective classification, but it is not evidence that anyone could have been warned
> in time.
>
> The honest early-warning test is `CUTOFF_DAY = 30`: rebuild the chain from the first 30 days
> only and score again. Whatever survives that truncation is signal you could actually have
> acted on. The gap between the two runs is the part that was hindsight.

In [ ]:
BASELINE_QUERY = '''
MATCH (s:Student)-[:WAS_REGISTERED]->(:StudentRegistration)-[cc:CONTAINS_COURSE]->(c:Course)
WHERE c.codeModule = $module
WITH DISTINCT s, cc.finalResult AS finalResult
// An inner match on purpose: a student with no interactions has no chain, so is
// not in the projection and not in any trained model. Scoring the baseline over
// a larger population than the models see would not be a comparison.
MATCH (s)-[r:REVIEWED_MATERIAL]->(:EducationalMaterial)<-[:HAS_MATERIAL]-(c2:Course)
WHERE c2.codeModule = $module
  AND ($cutoff IS NULL OR r.date <= $cutoff)
RETURN s.id AS studentId,
       CASE WHEN finalResult IN $passResults THEN 1 ELSE 0 END AS passed,
       sum(r.sumClick) AS clicks
'''
with driver.session(database=NEO4J_DATABASE) as session:
    base = (pd.DataFrame(session.run(
        BASELINE_QUERY, module=MODULE, cutoff=CUTOFF_DAY,
        passResults=PASS_RESULTS).data())
            .drop_duplicates(subset='studentId'))

def macro_f1(truth, predicted):
    scores = []
    for label in (0, 1):
        tp = ((predicted == label) & (truth == label)).sum()
        fp = ((predicted == label) & (truth != label)).sum()
        fn = ((predicted != label) & (truth == label)).sum()
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        scores.append(2 * precision * recall / (precision + recall)
                      if precision + recall else 0.0)
    return sum(scores) / len(scores)

predicted = (base['clicks'] >= base['clicks'].median()).astype(int)
BASELINE = {'F1_MACRO': macro_f1(base['passed'], predicted),
            'ACCURACY': (predicted == base['passed']).mean()}
majority = max(base['passed'].mean(), 1 - base['passed'].mean())

rows = [{'method': label, **scores, 'winner': winners[label]}
        for label, scores in results.items()]
rows.append({'method': 'clicks >= median (no model)', **BASELINE, 'winner': '-'})
rows.append({'method': 'always predict majority', 'F1_MACRO': None,
             'ACCURACY': majority, 'winner': '-'})
print(pd.DataFrame(rows).set_index('method').to_string())

both = results.get('journey + volume', {}).get('ACCURACY')
volume = results.get('volume only', {}).get('ACCURACY')
journey = results.get('journey embedding only', {}).get('ACCURACY')
if both and volume:
    gain = (both - volume) * 100
    print(f'\njourney embedding adds {gain:+.2f} accuracy points on top of volume')
    print('  within noise on this test split' if abs(gain) < 1.5
          else '  a real difference on this test split')
if journey:
    print(f'sequence alone reaches {journey:.4f} accuracy '
          f'against {majority:.4f} for always guessing the majority class')

## 13. Where the best model is wrong

A confusion matrix over every labelled student. For an early-warning use the interesting cell is
bottom-left: students who failed or withdrew and were *not* flagged.

> **These numbers include students the model trained on.** `predict_stream` covers every node with
> the target label, and GDS does not expose which nodes its internal split held back. On this data
> that inflated precision by up to 45 points at early cutoffs. Treat this matrix as a sanity check
> on the shape of the errors, and take the honest figures from step 14, which evaluates on a
> holdout it controls.

In [ ]:
best_label = max(results, key=lambda k: results[k].get('ACCURACY') or 0)
print(f'best variant: {best_label}')
best = models[best_label]

predictions = best.predict_stream(G, target_node_labels=['Student'])
truth = gds.graph.node_properties.stream(G, 'passed', node_labels=['Student'])
truth_column = [c for c in truth.columns if c != 'nodeId'][-1]
predicted_column = [c for c in predictions.columns
                    if c != 'nodeId' and 'probab' not in c.lower()][-1]

merged = (predictions.rename(columns={predicted_column: 'predicted'})[['nodeId', 'predicted']]
          .merge(truth.rename(columns={truth_column: 'actual'})[['nodeId', 'actual']],
                 on='nodeId'))
print()
print(pd.crosstab(merged['actual'], merged['predicted'],
                  rownames=['actual'], colnames=['predicted']).to_string())

caught = ((merged['actual'] == 0) & (merged['predicted'] == 0)).sum()
at_risk = (merged['actual'] == 0).sum()
print(f'\nat-risk students flagged: {caught:,} of {at_risk:,} '
      f'({caught / at_risk * 100:.1f}%)')
print(f"overall agreement: {(merged['predicted'] == merged['actual']).mean():.4f}")

## 14. Where does the signal start, and does it hold out of sample?

A single cutoff tells you whether a model works at that moment; it does not tell you when the
information arrives. This sweeps several cutoffs and reports one table.

**It evaluates on a holdout the model never saw.** That matters more than it sounds. GDS splits
internally but does not expose which nodes landed in its test set, so `predict_stream` over all
students mixes training data into the evaluation — and on this data that inflated precision by up
to **45 points**. The split here is ours: students carry an `isHoldout` flag, the pipeline trains
on a `TrainStudent` label and predicts on `HoldoutStudent`. Both figures are printed so the gap is
visible rather than assumed.

**Read precision and recall, not accuracy.** Accuracy counts every unflagged failure against the
model, which punishes a conservative classifier for being conservative. It also hides
memorisation: in the runs below the accuracy gap between in-sample and holdout was 3–21 points
while the precision gap reached 45.

**It rebuilds nothing.** The chain is ordered by day, so a *prefix by day* is exactly the chain a
truncated build produces — `tgt.day <= cutoff + SHIFT` is the whole trick. `logClicks`, the
threshold baseline and the majority floor are recomputed per window, so no cutoff is compared
against another's hindsight.

One caveat that remains: FastPath runs over the whole graph, holdout students included. It is
unsupervised and never sees a label, so this is transductive rather than leaky — but the embedding
was fitted knowing those students' *journeys*, if not their outcomes.

Requires `CUTOFF_DAY = None` in step 4, since it needs the whole chain to take prefixes of.

In [ ]:
RUN_SWEEP = False                       # set True to sweep
SWEEP_CUTOFFS = [30, 60, 90, None]      # None = the whole journey
HOLDOUT_FRACTION = 0.30
SPLIT_SEED = 42

PREFIX_PROJECTION = """
MATCH (src)-[r:FIRST_INTERACTION|NEXT_INTERACTION]->(tgt:Interaction)
WHERE tgt.module = $module AND tgt.day <= $maxDay
RETURN gds.graph.project.remote(src, tgt, {
    sourceNodeLabels: labels(src),
    targetNodeLabels: labels(tgt),
    sourceNodeProperties: src { .id, .day, .clicks, .activityTypeId, .features,
                                .passed, .logClicks,
                                isHoldout: CASE WHEN src.id IN $holdout
                                                THEN 1 ELSE 0 END },
    targetNodeProperties: tgt { .day, .clicks, .activityTypeId, .features },
    relationshipType: type(r)
})
"""
WINDOW_QUERY = """
MATCH (s:Student)-[:WAS_REGISTERED]->(:StudentRegistration)-[cc:CONTAINS_COURSE]->(c:Course)
WHERE c.codeModule = $module
WITH DISTINCT s, cc.finalResult AS finalResult
OPTIONAL MATCH (s)-[r:REVIEWED_MATERIAL]->(:EducationalMaterial)<-[:HAS_MATERIAL]-(c2:Course)
WHERE c2.codeModule = $module AND ($cutoff IS NULL OR r.date <= $cutoff)
WITH s, finalResult, coalesce(sum(r.sumClick), 0) AS clicks
SET s.passed = CASE WHEN finalResult IN $passResults THEN 1 ELSE 0 END,
    s.logClicks = log(toFloat(clicks) + 1.0)
"""

def _detection(frame):
    """Flagged / recall / precision for the at-risk class, which is 0."""
    at_risk = int((frame['act'] == 0).sum())
    flagged = int((frame['pred'] == 0).sum())
    caught = int(((frame['act'] == 0) & (frame['pred'] == 0)).sum())
    return {'n': len(frame), 'at_risk': at_risk, 'flagged': flagged,
            'recall': caught / at_risk if at_risk else 0.0,
            'precision': caught / flagged if flagged else 0.0,
            'accuracy': float((frame['pred'] == frame['act']).mean())}

def _evaluate(model, graph, node_label):
    preds = model.predict_stream(graph, target_node_labels=[node_label])
    truth = gds.graph.node_properties.stream(graph, 'passed', node_labels=[node_label])
    tcol = [c for c in truth.columns if c != 'nodeId'][-1]
    pcol = [c for c in preds.columns if c != 'nodeId' and 'probab' not in c.lower()][-1]
    return _detection(preds.rename(columns={pcol: 'pred'})[['nodeId', 'pred']]
                      .merge(truth.rename(columns={tcol: 'act'})[['nodeId', 'act']],
                             on='nodeId'))

if not RUN_SWEEP:
    print('RUN_SWEEP is False, so this cell did nothing.')
    print('Set it to True to measure where the predictive signal starts.')
elif CUTOFF_DAY is not None:
    print(f'The sweep takes prefixes of the whole chain, but CUTOFF_DAY is {CUTOFF_DAY}, '
          'so the chain on disk is already truncated.')
    print('Set CUTOFF_DAY = None in step 4, rebuild, then re-run this cell.')
else:
    import random

    with driver.session(database=NEO4J_DATABASE) as session:
        chained = sorted(r['studentId'] for r in session.run(
            'MATCH (s:Student)-[:FIRST_INTERACTION]->(:Interaction {module: $module}) '
            'RETURN DISTINCT s.id AS studentId', module=MODULE))
    shuffled = chained[:]
    random.Random(SPLIT_SEED).shuffle(shuffled)
    HOLDOUT = sorted(shuffled[:int(len(shuffled) * HOLDOUT_FRACTION)])
    print(f'{len(chained):,} students with a chain -> '
          f'{len(chained) - len(HOLDOUT):,} train / {len(HOLDOUT):,} holdout '
          f'(seed {SPLIT_SEED})')

    rows = []
    for cutoff in SWEEP_CUTOFFS:
        tag = 'full' if cutoff is None else f'd{cutoff}'
        max_day = 10 ** 9 if cutoff is None else cutoff + SHIFT
        last_day = span['maxDate'] if cutoff is None else cutoff
        observation = float(last_day + SHIFT + 1)
        horizon = int(observation) + 10
        print(f'\n=== cutoff {tag} (days <= {max_day}) ===', flush=True)

        with driver.session(database=NEO4J_DATABASE) as session:
            session.run(WINDOW_QUERY, module=MODULE, cutoff=cutoff,
                        passResults=PASS_RESULTS).consume()

        gds.graph.project.cypher(
            graph_name=f'sweep-{tag}', query=PREFIX_PROJECTION,
            query_parameters={'module': MODULE, 'maxDay': max_day, 'holdout': HOLDOUT},
            overwrite=True)
        Gs = gds.graph.get(f'sweep-{tag}')

        gds.fast_path.mutate(
            Gs, base_node_label='Student', event_node_label='Interaction',
            mutate_property='journeyEmbedding', embedding_dimension=128,
            lookback_horizon=horizon, num_time_anchors=NUM_TIME_ANCHORS,
            event_node_categorical_properties=['activityTypeId'],
            event_node_feature_vector_property='features',
            event_node_time_property='day',
            first_relationship_type='FIRST_INTERACTION',
            next_relationship_type='NEXT_INTERACTION',
            observation_time=observation, smoothing_window=2,
            smoothing_rate=10.0 / horizon, random_seed=42)

        # Label AFTER fast_path, never before. fast_path.mutate registers
        # journeyEmbedding against base_node_label='Student' only, so a
        # TrainStudent label that existed at projection time would not carry it
        # and training fails with "node properties do not exist in the graph".
        for split_label, node_filter in [
            ('TrainStudent', 'n:Student AND n.isHoldout = 0'),
            ('HoldoutStudent', 'n:Student AND n.isHoldout = 1'),
        ]:
            gds.graph.node_labels.mutate(Gs, split_label, node_filter=node_filter)
        if 'journeyEmbedding' not in set(Gs.node_properties().get('TrainStudent', [])):
            raise SystemExit('TrainStudent does not carry journeyEmbedding; check the '
                             'labelling order against the comment above.')

        n_train = len(gds.graph.node_properties.stream(
            Gs, 'passed', node_labels=['TrainStudent']))
        n_hold = len(gds.graph.node_properties.stream(
            Gs, 'passed', node_labels=['HoldoutStudent']))
        print(f'  {Gs.node_count():,} nodes, {Gs.relationship_count():,} rels | '
              f'train {n_train:,} / holdout {n_hold:,}', flush=True)

        row = {'cutoff': tag}
        for label, features in [('volume', ['logClicks']),
                                ('both', ['journeyEmbedding', 'logClicks'])]:
            names = (f'sweep-pipe-{tag}-{label}', f'sweep-model-{tag}-{label}')
            for drop in (lambda: gds.model.get(names[1]).drop(),
                         lambda: gds.pipeline.node_classification.get(names[0]).drop()):
                try:
                    drop()
                except Exception:
                    pass
            pipe, _ = gds.pipeline.node_classification.create(names[0])
            pipe.select_features(features)
            pipe.configure_split(test_fraction=0.2, validation_folds=4)
            pipe.add_logistic_regression(penalty=(0.001, 1.0), max_epochs=300)
            pipe.add_random_forest(max_depth=(4, 16), number_of_decision_trees=200)
            model, _ = pipe.train(Gs, model_name=names[1],
                                  metrics=['F1_MACRO', 'ACCURACY'],
                                  target_property='passed',
                                  target_node_labels=['TrainStudent'],
                                  random_seed=42)

            out = _evaluate(model, Gs, 'HoldoutStudent')
            ins = _evaluate(model, Gs, 'TrainStudent')
            row.update({f'{label}_at_risk': out['at_risk'],
                        f'{label}_flagged': out['flagged'],
                        f'{label}_recall': out['recall'],
                        f'{label}_precision': out['precision'],
                        f'{label}_accuracy': out['accuracy'],
                        f'{label}_insample_precision': ins['precision']})
            print(f"    {label:7s} HOLDOUT flagged {out['flagged']:4,} of "
                  f"{out['at_risk']:,} | recall {out['recall']:.3f} "
                  f"precision {out['precision']:.3f} acc {out['accuracy']:.3f}")
            print(f"    {'':7s} in-sample precision {ins['precision']:.3f} "
                  f"(gap {ins['precision'] - out['precision']:+.3f})", flush=True)
            for drop in (lambda: gds.model.get(names[1]).drop(),
                         lambda: gds.pipeline.node_classification.get(names[0]).drop()):
                try:
                    drop()
                except Exception:
                    pass
        rows.append(row)
        try:
            Gs.drop()
        except Exception:
            pass

    sweep = pd.DataFrame(rows).set_index('cutoff')
    print('\n' + '=' * 76)
    print('OUT-OF-SAMPLE at-risk detection, journey + volume')
    print('=' * 76)
    print(sweep[['both_at_risk', 'both_flagged', 'both_recall', 'both_precision',
                 'both_accuracy']].round(4).to_string())
    print('\nvolume only, same holdout')
    print(sweep[['volume_flagged', 'volume_recall', 'volume_precision',
                 'volume_accuracy']].round(4).to_string())
    print('\nhow much an in-sample-inclusive evaluation would have overstated precision')
    print((sweep[['both_insample_precision', 'volume_insample_precision']]
           .rename(columns={'both_insample_precision': 'both',
                            'volume_insample_precision': 'volume'})
           .sub(sweep[['both_precision', 'volume_precision']]
                .rename(columns={'both_precision': 'both',
                                 'volume_precision': 'volume'}))
           ).round(4).to_string())

## 15. Train the recommended model and persist it to the Aura model catalog

The configuration this repository recommends: **`journeyEmbedding + logClicks + submission`, cut
at day 90.** It is the top-precision arm in both measured modules — 0.855 on BBB, 0.832 on GGG —
and `docs/assessment-submission.md` has the ablation behind that.

A session model normally dies with the session, which has a 2-hour TTL. `gds.model.store()`
persists it beyond that so `aga_score_unseen_module.ipynb` can load it and score a module this
notebook never trained on.

Two things make the stored model transferable, and both are easy to lose:

**The activity-type vocabulary must be global.** Step 4 enumerates all 20 types across the whole
dataset rather than the 7 in GGG, so `activityTypeId` means the same thing in every module. With a
per-module vocabulary the stored model would score other modules confidently and meaninglessly.

**The submission features must be scale-free.** `submissionRate` is a ratio and `missedAll` /
`missedFirst` are booleans, so they carry across modules with different assessment counts.
`meanLateness` is in days, which is comparable.

A guard refuses to store a degenerate model. GDS's pipeline can return a classifier that predicts
the majority class everywhere and reports it as a successful train — it is reproducible on this
data with FastRP — so the model is required to flag *something* on the training module before it is
allowed into the catalog.

In [ ]:
RECOMMENDED_CUTOFF = 90
MODEL_NAME = f'oulad-atrisk-d{RECOMMENDED_CUTOFF}'
PIPELINE_NAME = 'oulad-atrisk-pipeline'
RECOMMENDED_FEATURES = [EMBEDDING_PROPERTY, 'logClicks',
                        'submissionRate', 'missedAll', 'missedFirst', 'meanLateness']

if CUTOFF_DAY != RECOMMENDED_CUTOFF:
    print(f'CUTOFF_DAY is {CUTOFF_DAY}, not {RECOMMENDED_CUTOFF}.')
    print('The recommendation is specifically the day-90 model: the untruncated one reads '
          'when activity stopped, which for a withdrawal is the label restated.')
    print(f'Set CUTOFF_DAY = {RECOMMENDED_CUTOFF} in step 4, rebuild, then run this cell.')
elif got['assessmentsDue'] < 3:
    print(f"Only {got['assessmentsDue']} assessment(s) are due in {MODULE} by day "
          f'{RECOMMENDED_CUTOFF}, so submissionRate is effectively binary, missedAll and '
          'missedFirst are the same column, and meanLateness comes from a single item.')
    print('Four of the six features collapse to one boolean, which is not the configuration '
          'this repository recommends and not a model worth transferring.')
    print("Re-run with OULAD_MODULE='BBB' (15 assessments due by day 90) or another "
          'assessment-dense module.')
else:
    try:
        gds.model.drop(MODEL_NAME, fail_if_missing=False)
    except Exception:
        pass
    try:
        gds.pipeline.node_classification.get(PIPELINE_NAME).drop()
    except Exception:
        pass

    pipe, _ = gds.pipeline.node_classification.create(PIPELINE_NAME)
    pipe.select_features(RECOMMENDED_FEATURES)
    pipe.configure_split(test_fraction=0.3, validation_folds=4)
    pipe.add_logistic_regression(penalty=(0.001, 1.0), max_epochs=300)
    pipe.add_random_forest(max_depth=(4, 16), number_of_decision_trees=200)

    recommended, _ = pipe.train(
        G, model_name=MODEL_NAME, metrics=['F1_MACRO', 'ACCURACY'],
        target_property='passed', target_node_labels=['Student'], random_seed=42)

    scores = {m: v.get('test') for m, v in (recommended.metrics() or {}).items()
              if isinstance(v, dict)}
    print(f'trained {MODEL_NAME} on {MODULE}: {scores}')
    print('winner:', (recommended.best_parameters() or {}).get('methodName', '?'))

    # A model that flags nobody is a fitting failure that reports as a success.
    check = recommended.predict_stream(G, target_node_labels=['Student'])
    pcol = [c for c in check.columns
            if c != 'nodeId' and 'probab' not in c.lower()][-1]
    flagged = int((check[pcol] == 0).sum())
    print(f'flags {flagged:,} of {len(check):,} students on the training module')
    if flagged == 0:
        raise SystemExit(
            'The trained model predicts the majority class for every student. That is a '
            'degenerate fit, not a model, and storing it would put a constant classifier '
            'in the catalog. Do not store it.')

    stored = gds.model.store(MODEL_NAME)
    print('\nstored:', stored)
    print('catalog now holds:', [m.model_name for m in gds.model.list()])

    # The scoring notebook needs these to rebuild features identically.
    print('\n--- copy these into aga_score_unseen_module.ipynb, step 3 ---')
    print(f'MODEL_NAME     = {MODEL_NAME!r}')
    print(f'ASSESSMENTS    = {got["assessmentsDue"]} due by day {RECOMMENDED_CUTOFF}')
    print(f'TRAINED_ON     = {MODULE!r}')
    print(f'CUTOFF_DAY     = {RECOMMENDED_CUTOFF}')
    print(f'FEATURES       = {RECOMMENDED_FEATURES}')
    print(f'TYPE_IDS       = {TYPE_IDS}')

## 16. Clean up

Three things to release, and all of them matter: the trained models and pipelines, the session
(billed compute), and the event chain plus the two `Student` properties, which are the only
marks this notebook leaves on your database.

`DELETE_CHAIN` defaults to `True`. A property on existing nodes is cheap to leave behind; a few
hundred thousand extra nodes is not. Set it to `False` to keep the chain for another run — step
5 skips students that already have one.

In [ ]:
def db_execute(query, **params):
    """Run a write, rebuilding the driver if its connection has gone stale."""
    global driver
    for attempt in (1, 2):
        try:
            with driver.session(database=NEO4J_DATABASE) as session:
                return session.run(query, **params).single()
        except Exception as error:
            if attempt == 2:
                raise
            print(f'  reconnecting after: {str(error)[:70]}')
            try:
                driver.close()
            except Exception:
                pass
            driver = GraphDatabase.driver(
                NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD),
                liveness_check_timeout=30, max_connection_lifetime=600)

DELETE_CHAIN = True

for label, (_, suffix) in VARIANTS.items():
    for what, drop in [
        ('model', lambda s=suffix: gds.model.drop(f'fastpath-model-{s}',
                                                  fail_if_missing=False)),
        ('pipeline', lambda s=suffix: gds.pipeline.node_classification.get(
            f'fastpath-pipeline-{s}').drop()),
    ]:
        try:
            drop()
        except Exception as error:
            print(f'{what} {suffix}: {str(error)[:70]}')
for what, drop in [
    ('model', lambda: gds.model.drop(MODEL_NAME, fail_if_missing=False)),
    ('pipeline', lambda: gds.pipeline.node_classification.get(PIPELINE_NAME).drop()),
]:
    try:
        drop()
    except Exception as error:
        print(f'{what} {MODEL_NAME}: {str(error)[:70]}')
print('models and pipelines released')
print('NOTE: gds.model.store() persists beyond the session, and dropping the in-session '
      'copy does not delete the stored one -- that is the point. Remove it with '
      f'gds.model.delete({MODEL_NAME!r}) if you want it gone.')

try:
    G.drop(); print('projection dropped')
except Exception as error:
    print('projection:', error)
try:
    gds.delete(); print('session deleted')
except Exception as error:
    print('session:', error)

if DELETE_CHAIN:
    # Batched: DETACH DELETE over a few hundred thousand nodes in one
    # transaction is how you run a session out of memory.
    DELETE_QUERY = '''
    MATCH (i:Interaction {module: $module})
    WITH i LIMIT $batch
    DETACH DELETE i
    RETURN count(*) AS deleted
    '''
    removed = 0
    while True:
        deleted = db_execute(DELETE_QUERY, module=MODULE, batch=10000)['deleted']
        removed += deleted
        if deleted:
            print(f'  deleted {removed:,}', flush=True)
        if deleted == 0:
            break
    print(f'removed {removed:,} interaction nodes')
    db_execute('MATCH (s:Student) WHERE s.passed IS NOT NULL '
               'REMOVE s.passed, s.logClicks RETURN 0 AS done')
    db_execute('MATCH (s:Student) WHERE s.submissionRate IS NOT NULL '
               'REMOVE s.submissionRate, s.missedAll, s.missedFirst, '
               's.meanLateness RETURN 0 AS done')
    print('removed the passed, logClicks and submission properties from Student')
else:
    print('DELETE_CHAIN is False; the event chain is still in the database')

with driver.session(database=NEO4J_DATABASE) as session:
    left = session.run('MATCH (i:Interaction) RETURN count(i) AS n').single()['n']
    totals = session.run(
        'MATCH (n) WITH count(n) AS nodes '
        'MATCH ()-[r]->() RETURN nodes, count(r) AS relationships').single()
driver.close()

print(f'\ninteraction nodes remaining: {left:,}')
print(f"graph totals: {totals['nodes']:,} nodes, {totals['relationships']:,} relationships")
print('(66,920 nodes and 8,818,076 relationships is the untouched OULAD graph)')

## What the runs showed

Module GGG. FastPath embeddings into a node classification pipeline, evaluated on a **707-student
holdout the model never trained on** (30%, seed 42).

### Out-of-sample at-risk detection — journey + volume

Class 0 is fail or withdrawn. **Recall** is the share of eventual failures flagged; **precision**
is the share of flags that are real.

| cutoff | at risk | flagged | recall | precision | accuracy |
| --- | --- | --- | --- | --- | --- |
| day 30 | 234 | 112 | 0.231 | 0.482 | 0.635 |
| **day 90** | 258 | 133 | **0.372** | **0.722** | 0.717 |
| whole journey | 261 | 204 | 0.759 | 0.971 | 0.902 |

Volume alone on the same holdout: precision **0.382 / 0.417 / 0.825**. The embedding still nearly
doubles it at day 90, which is the claim that survives an honest split.

### What an in-sample-inclusive evaluation would have told you

| cutoff | precision gap | recall gap | accuracy gap |
| --- | --- | --- | --- |
| day 30 | **+0.450** | +0.356 | +0.206 |
| day 90 | +0.165 | +0.144 | +0.090 |
| whole journey | +0.017 | +0.068 | +0.033 |

**Day 30 was almost entirely memorisation** — 0.932 precision in-sample against 0.482 held out. An
earlier version of this notebook recommended it as a small, high-precision worklist. It is not
one.

Note also that the accuracy gap is 3–21 points while the precision gap reaches 45. Accuracy is
dominated by the majority class and hides memorisation; arguing "the accuracy gap is small, so the
inflation is small" is invalid, and was the reasoning that let the day-30 claim stand.

### The same procedure on a second module

BBB — 6,484 students, 1,139,085 events, same seed and holdout fraction, 1,938 held-out students at
day 90:

| cutoff | features | flagged | of at risk | recall | precision | precision gap |
| --- | --- | --- | --- | --- | --- | --- |
| day 30 | volume | 549 | 812 | 0.425 | 0.628 | −0.015 |
| day 30 | both | 446 | 812 | 0.386 | 0.702 | +0.141 |
| day 90 | volume | 591 | 826 | 0.504 | 0.704 | −0.010 |
| **day 90** | **both** | **520** | **826** | **0.528** | **0.838** | **+0.015** |
| whole journey | volume | 735 | 829 | 0.679 | 0.766 | −0.011 |
| whole journey | both | 677 | 829 | 0.776 | 0.950 | +0.023 |

Day 90 replicates and improves — 0.838 against GGG's 0.722, with almost no inflation. **GGG's
day-30 collapse does not generalise**: on BBB day 30 holds out at 0.702, so when the signal arrives
is a property of the presentation, not of the method.

Note also that every negative gap belongs to a volume-only model and every positive one to a model
carrying `journeyEmbedding`. One logged feature cannot memorise 4,500 training rows; 129 features
can. Assume any embedding figure quoted without a holdout is inflated.

### What to take from it

**Day 90, journey + volume, is the recommendation.** Precision 0.722 on GGG and 0.838 on BBB. It
flags *fewer* students than a click-volume model and catches *more* of the failures in both
modules — 133 flags catching 96 against volume's 163 catching 68 on GGG, 520 catching 436 against
591 catching 416 on BBB.

**The whole-journey model is not a predictor.** 0.971 precision, but it reads *when activity
stopped*, which for a withdrawal is the label restated. By the time it is confident the
presentation is over and `finalResult` is already in the data.

**Volume alone is not a fallback early.** 0.382 to 0.417 on GGG and 0.628 to 0.704 on BBB across
days 30 to 90, against 0.482–0.722 and 0.702–0.838 for the embedding. The interpretable single
feature works retrospectively and is beaten when it would matter — though it is the one baseline
that never needed a holdout correction, so it stays the honest floor.

## Why modularity was the wrong measure too

Two earlier runs were judged on Louvain modularity — 0.5470 and 0.5122 — and read as failure.
Modularity scores how cleanly a similarity graph partitions, a different question again. Three
measures, three verdicts on the same embedding: modularity said it failed, accuracy said it was
mediocre, in-sample precision said it was excellent, and an honest holdout says it is useful at
day 90 and not before. Pick the measure that matches the decision, then check the evaluation is
clean.

## Where to look next

`MODULE` in step 4 — GGG and BBB are measured, AAA through FFF are not. They agree on the
recommendation and differ by 12 precision points on its value, so re-measure rather than carrying a
number across. `SWEEP_CUTOFFS` takes any list.

For an operational system the highest-value trigger is probably not a model: students who never
touch a material cannot be projected or classified, and in module BBB 88.1% of that group
withdrew. A rule as simple as "no activity by day 14" flags a higher-risk population than any
classifier here, and this model belongs after it.